# 실습 13: 답안지를 덮고 풀어보기
- 상황: 검사 결과가 없는 라인이 들어온다. 답 없이 이상한 것을 찾을 수 있나
- 목표: 정답 열을 가리고 이상을 지목한 뒤, 마지막에만 답을 꺼내 채점한다

## Step 0. 정제본 불러오기

In [1]:
import pandas as pd

# 1. 정제본 불러오기 (day02 실습 결과물)
df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

# 2. 센서 열의 빈칸을 그 열의 중앙값으로 채우기
센서열 = [c for c in df.columns if c.startswith("sensor_")]
df[센서열] = df[센서열].fillna(df[센서열].median())

# 3. 입력은 센서 열만
X = df[센서열]

# 4. 정답은 따로 만들어만 둔다 - 이상탐지는 정답을 보지 않고 학습한다
#    아래 실습에서는 쓰지 않고, 나중에 채점할 때만 꺼내 쓴다
정답 = (df["result"] == "불량").astype(int)

print("전체 행 수:", len(df))
print("센서 열 개수:", len(센서열))
print("불량 건수:", int(정답.sum()), "건")
print("불량 비율:", round(정답.mean() * 100, 2), "%")

전체 행 수: 1567
센서 열 개수: 50
불량 건수: 104 건
불량 비율: 6.64 %


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 답 없이 찾는 말

| 말 | 뜻 |
|---|---|
| 지도학습 | 정답이 붙어 있는 기록으로 배우는 것. 나흘 동안 한 것이 전부 이것 |
| 비지도학습 | 정답 없이 데이터의 생김새만 보고 하는 것. 오늘 넘는 선 |
| 이상탐지 | 다수와 다른 것을 찾아내는 일. 정답 대신 동료들과 비교한다 |
| 고립시키기 | 몇 번 잘라야 그 줄만 혼자 남나를 보는 방법. 빨리 혼자가 되면 이상 |
| 이상 비율 | 전체 중 몇 %를 이상으로 볼지 사람이 정해주는 값 |

## Step 2. 답을 가리고 이상한 것 지목하기

In [2]:
# 다수와 다른 줄을 찾아주는 도구를 불러온다
from sklearn.ensemble import IsolationForest

# ① 바로 위에서 불러온 도구 이름을 그대로 쓴다
# ② 전체의 몇 %를 이상으로 볼지 내가 정해서 넣는 자리. 0.05 는 5%
# ③ 다시 돌려도 같은 결과가 나오게 번호를 고정하는 자리 - 어제도 셬다
탐지기 = IsolationForest(contamination=0.05, random_state=42)

# ④ 정답은 넣지 않는다 - 센서 값만 담긴 것을 넣는다. 이것이 오늘의 핵심
지목 = 탐지기.fit_predict(X)

# ⑤ 돌려주는 값은 두 가지뿐이다. 이상이면 무엇으로 나오나
print("이상이라 지목한 건수:", (지목 == -1).sum())
print("전체 건수:", len(X))

이상이라 지목한 건수: 79
전체 건수: 1567


### 문법 노트 - 답 없이 지목하기

| 밑줄 | 넣은 것 | 하는 일 | 새로 배운 것인가 |
|---|---|---|---|
| ① | `IsolationForest` | 몇 번 잘라야 혼자 남는지로 이상을 찾는 도구 | 새로 배움 |
| ② | `contamination` | 전체의 몇 %를 이상으로 볼지 정하는 자리 | 새로 배움 |
| ③ | `random_state` | 다시 돌려도 같게 나오도록 고정하는 번호 | 나눠 때·모델 만들 때 계속 셬다 |
| ④ | `X` | 센서 값만 담긴 것. 정답은 안 들어간다 | 계속 쓴 이름 |
| ⑤ | `-1` | 이상이라는 표시 (정상은 `1`) | 새로 배움 · 0과 1이 아니다 |

## Step 4. 답을 꺼내 채점하기

In [3]:
# ⑥ 지목이 이상인 자리만 참이 되는 표시를 만든다
이상표시 = (지목 == -1)

# ⑦ 두 표시가 둘 다 참인 자리만 남기는 기호
# ⑧ 불량을 무엇으로 적어된는지
# ⑨ 참인 자리가 몇 개인지 세는 것
잡은것 = (이상표시 & (정답 == 1)).sum()

# 지목 중 진짜 비율 = 잡은 것 나누기 지목한 건수
지목중진짜 = 잡은것 / 이상표시.sum()

# 전체 불량 중 잡은 비율 = 잡은 것 나누기 전체 불량 건수
전체중잡은것 = 잡은것 / (정답 == 1).sum()

print("지목", 이상표시.sum(), "건 / 그중 불량", 잡은것, "건")
print("지목 중 진짜:", round(지목중진짜 * 100, 1), "%")
print("전체 불량 중 잡은 것:", round(전체중잡은것 * 100, 1), "%")

지목 79 건 / 그중 불량 12 건
지목 중 진짜: 15.2 %
전체 불량 중 잡은 것: 11.5 %


### 문법 노트 - 채점할 때 쓴 것

| 밑줄 | 넣은 것 | 하는 일 | 새로 배운 것인가 |
|---|---|---|---|
| ⑥ | `-1` | 이 도구가 쓰는 이상 표시 | 위에서 한 번 나왔다 |
| ⑦ | `&` | 양쪽이 다 참인 자리만 참 | 새로 배움 |
| ⑧ | `1` | 불량을 1로 적어둔 그 값 | 사흘째 쓰는 약속 |
| ⑨ | `.sum()` | 참인 자리의 개수를 센다 | 참·거짓에 쓰면 개수가 나온다 |

## Step 5. 아무 근거 없이 찍었다면

In [4]:
# 이상탐지를 쓰지 않고, 그냥 무작위로 79건을 골라본다
# sample - 아무 기준 없이 뽑기, random_state=42 로 고정해서 매번 같은 79건이 나오게 한다
무작위지목 = 정답.sample(n=79, random_state=42)

잡은불량 = int(무작위지목.sum())
지목중진짜 = 잡은불량 / 79 * 100      # 지목한 79건 중 몇 %가 진짜 불량이었나
전체불량률 = 정답.mean() * 100        # 아무것도 안 골랐을 때의 원래 불량 비율

print("지목한 건수: 79 건")
print("그중 불량:", 잡은불량, "건")
print("지목 중 진짜:", round(지목중진짜, 2), "%")
print("전체 불량률:", round(전체불량률, 2), "%")
print("차이:", round(지목중진짜 - 전체불량률, 2), "%p")

지목한 건수: 79 건
그중 불량: 5 건
지목 중 진짜: 6.33 %
전체 불량률: 6.64 %
차이: -0.31 %p


## Step 6. 오늘 알게 된 것

| 고른 방법 | 지목 건수 | 그중 불량 | 지목 중 진짜 |
|---|---|---|---|
| 아무 근거 없이 | [79] | [5] | [6.3%] |
| 고립시키기 (5%) | [79] | [12] | [15.2%] |

- 정답을 안 쓰고 얻은 것 : [같은 79건을 고르는데 불량이 두 배 넘게 들어 있었다]
- 아직 못 미더운 점 : [전체 불량 104건 중 12건만 잡았다. 92건은 그대로 지나갔다]
- 이 결과를 어디에 쓸 수 있나 : [먼저 열어볼 순서를 정하는 데. 불량 판정으로 쓰면 안 된다]

---
## 직접 해보기 (도전) - 몇 %로 볼 것인가

- 상황: 5%로 봤는데, 2%나 10%로 하면 어떻게 될까
- 할 일: 이상 비율을 바꿔가며 지목 건수와 적중을 비교한다
- 결과물: 네 줄짜리 표 1개 + 한 줄 메모

In [5]:
# 앞에서 쓴 도구를 그대로, 이상 비율만 바꿔가며 네 번 돌린다
from sklearn.ensemble import IsolationForest

전체불량 = int((정답 == 1).sum())

print("이상 비율 | 지목 건수 | 그중 불량 | 지목 중 진짜 | 전체 불량 중 잡은 것")
print("-" * 66)

for 이상비율 in [0.02, 0.05, 0.10, 0.15]:
    # contamination 자리만 바꾸고 나머지는 전부 같게 둔다
    탐지기 = IsolationForest(contamination=이상비율, random_state=42)

    # 지목할 때는 X 만 넣는다 - 정답은 여기 들어가지 않는다
    지목 = 탐지기.fit_predict(X)

    이상표시 = (지목 == -1)
    지목건수 = int(이상표시.sum())

    # 여기서부터가 채점 - 이제야 정답을 꺼낸다
    잡은것 = int((이상표시 & (정답 == 1)).sum())
    지목중진짜 = 잡은것 / 지목건수 * 100
    전체중잡은것 = 잡은것 / 전체불량 * 100

    print(f"{이상비율:>8.0%} | {지목건수:>6} 건 | {잡은것:>5} 건 | {지목중진짜:>9.1f} % | {전체중잡은것:>12.1f} %")

이상 비율 | 지목 건수 | 그중 불량 | 지목 중 진짜 | 전체 불량 중 잡은 것
------------------------------------------------------------------
      2% |     32 건 |     8 건 |      25.0 % |          7.7 %
      5% |     79 건 |    12 건 |      15.2 % |         11.5 %


     10% |    157 건 |    22 건 |      14.0 % |         21.2 %

     15% |    235 건 |    37 건 |      15.7 % |         35.6 %


### 문턱을 바꾸면

| 이상 비율 | 지목 건수 | 지목 중 진짜 | 전체 중 잡은 것 |
|---|---|---|---|
| 2% | [32] | [25.0%] | [7.7%] |
| 5% | [79] | [15.2%] | [11.5%] |
| 10% | [157] | [14.0%] | [21.2%] |
| 15% | [235] | [15.7%] | [35.6%] |

- 알게 된 것 : [많이 지목할수록 전체 불량 중 잡는 비율은 오르는데, 지목 중 진짜 비율은 25%에서 14%로 떨어진다. 어제 문턱을 옮길 때와 같은 모양이다]

---
## 더 해보기 - 지목 대신 점수로 줄 세우기

- 상황: 이상이냐 아니냐 두 갈래 말고, 이상한 순서대로 줄을 세우고 싶다
- 할 일: `score_samples` 로 점수를 꺼내고, 점수가 낮을수록 이상한 게 맞는지 먼저 확인한다
- 결과물: 세 줄짜리 표 1개

In [7]:
import numpy as np
from sklearn.ensemble import IsolationForest

# 바로 위 반복문에서 탐지기가 15%짜리로 바뀌었으므로, 5%짜리를 다시 만든다
탐지기5 = IsolationForest(contamination=0.05, random_state=42)
탐지기5.fit(X)                    # 여기도 정답은 넣지 않는다

# score_samples - 줄마다 점수를 하나씩 내준다. -1/1 이 아니라 실수 값이다
이상점수 = 탐지기5.score_samples(X)
지목5 = 탐지기5.predict(X)

print("[확인] 점수가 낮을수록 이상한 게 맞나")
print("점수 범위:", round(이상점수.min(), 4), "~", round(이상점수.max(), 4))
print("이상이라 지목한 것의 평균 점수:", round(이상점수[지목5 == -1].mean(), 4))
print("정상이라 본 것의 평균 점수  :", round(이상점수[지목5 == 1].mean(), 4))

# 도구가 이상이라 한 79건과, 점수가 가장 낮은 79건이 같은 줄인지 견줘본다
이상이라한줄 = set(np.flatnonzero(지목5 == -1))
점수낮은79줄 = set(np.argsort(이상점수)[:79])
print("이상이라 한 79건 = 점수가 가장 낮은 79건 ?", 이상이라한줄 == 점수낮은79줄)
print()

# 점수가 낮은 것부터 줄을 세운다 - 가장 이상한 것이 맨 앞
순서 = np.argsort(이상점수)
정답값 = 정답.to_numpy()
전체불량 = int(정답값.sum())

print("상위 몇 건 | 그중 불량 | 지목 중 진짜 | 전체 불량 중 잡은 것")
print("-" * 56)
for 상위 in [20, 50, 79]:
    뽑은줄 = 순서[:상위]
    잡은것 = int(정답값[뽑은줄].sum())      # 여기서만 정답을 꺼낸다
    print(f"{상위:>7} 건 | {잡은것:>5} 건 | {잡은것 / 상위 * 100:>9.1f} % |"
          f" {잡은것 / 전체불량 * 100:>12.1f} %")

[확인] 점수가 낮을수록 이상한 게 맞나
점수 범위: -0.648 ~ -0.3458
이상이라 지목한 것의 평균 점수: -0.5176
정상이라 본 것의 평균 점수  : -0.3958
이상이라 한 79건 = 점수가 가장 낮은 79건 ? True

상위 몇 건 | 그중 불량 | 지목 중 진짜 | 전체 불량 중 잡은 것
--------------------------------------------------------
     20 건 |     6 건 |      30.0 % |          5.8 %
     50 건 |     9 건 |      18.0 % |          8.7 %
     79 건 |    12 건 |      15.2 % |         11.5 %


### 가장 이상한 것부터 열어본다면

| 상위 몇 건 | 그중 불량 | 지목 중 진짜 |
|---|---|---|
| 20건 | [6] | [30.0%] |
| 50건 | [9] | [18.0%] |
| 79건 | [12] | [15.2%] |

- 알게 된 것 : [위에서부터 열어볼수록 적중이 높다. 상위 20건은 30.0%로 79건 전체(15.2%)의 두 배다]